# Second-dataset acquisition & screening

**Read this first — Daxing is NOT directly downloadable.**

The only official distribution is a **Baidu Pan** link from the GitHub README
(`pan.baidu.com`, fetch code `2024`). Baidu requires an account, throttles
non-VIP downloads hard, and actively blocks scripted access — a Colab
`wget`/`gdown` will not work. Options, in order of practicality:

| Option | Effort | Notes |
|---|---|---|
| **A. Dresden Kodak_M1063** | zero — already on your Drive | **5 same-model bodies**, 438–571 imgs each. Dresden also has 5-instance models in Casio EX-Z150, Nikon CoolPixS710, Olympus mju 1050SW, Praktica DCZ5.9, Ricoh GX100. Gives the multi-device/adapter replication the review says matters most, and empirical kappa_model from held-out bodies. Does NOT give a second sensor era |
| **B. FODB (Forchheim)** | low — direct HTTP from FAU | scriptable; but only two models have >1 device (Galaxy A6, Huawei P9 lite) and ~143 native imgs/device |
| **C. VISION** | low — direct HTTP from UNIFI | scriptable; ~335 native imgs/device average; same-model pairs are few — run the screener below |
| **D. Daxing via Baidu** | high | account + hours of throttled download + manual upload to Drive |
| **E. Email the authors** | low effort, days of latency | Tian/Xiao et al., BJTU / PPSUC. Frequently effective for academic redistribution |

**Recommendation:** start Option A today (zero friction, and it directly answers the
review's §6 "more devices and adapters" requirement). Kick off D or E in the background
if you want the second sensor era for the TIFS version.

In [ ]:
# %% 0 — mount + config
from google.colab import drive; drive.mount("/content/drive", force_remount=False)
import os, re, glob, json, subprocess, sys
from collections import Counter
DRIVE = "/content/drive/MyDrive"
DEST  = os.path.join(DRIVE, "forensic_datasets")
os.makedirs(DEST, exist_ok=True)
print("free on Drive:"); subprocess.run(["df","-h",DRIVE])

## 1. Universal same-model screener
Run on any dataset root. Prints camera models that have >= 2 physical devices and how many images each has, so you can size the splits before committing GPU time.

In [ ]:
# %% 1 — screener: which models have multiple devices, and enough images?
# Split rules per dataset. dev_of() -> device id; model_of() -> camera model.
RULES = {
  # Dresden:  Nikon_D200_0_12345.JPG  -> model Nikon_D200, device Nikon_D200_0
  "dresden": dict(
      rx=r"^(?P<model>.+)_(?P<inst>\d+)_(?P<shot>\d+)\.(jpe?g|JPE?G|png|PNG)$",
      dev=lambda m: f"{m['model']}_{m['inst']}", mod=lambda m: m["model"]),
  # Daxing:   device code = 5 chars; first 2 = model, last 2 = individual device
  "daxing": dict(
      rx=r"^(?P<code>[A-Za-z0-9]{4,6})[_-]?(?P<shot>\d+)\.(jpe?g|JPE?G)$",
      dev=lambda m: m["code"], mod=lambda m: m["code"][:-2]),
  # VISION:   D01_Samsung_GalaxyS3Mini/images/nat/D01_I_nat_0001.jpg
  "vision": dict(
      rx=r"^(?P<dev>D\d+)_I_(?P<kind>\w+?)_(?P<shot>\d+)\.(jpe?g|JPE?G)$",
      dev=lambda m: m["dev"], mod=None),      # model comes from the folder name
  # FODB:     D01_<Model>/...
  "fodb": dict(
      rx=r"^(?P<dev>D\d+)_.*\.(jpe?g|JPE?G)$",
      dev=lambda m: m["dev"], mod=None),
}

def screen(root, kind, need=340, folder_model=True):
    """List camera models with >=2 devices and the image count per device."""
    r = RULES[kind]; rx = re.compile(r["rx"]); dv = {}
    for fp in glob.glob(os.path.join(root, "**", "*.*"), recursive=True):
        m = rx.match(os.path.basename(fp))
        if not m: continue
        d = r["dev"](m)
        if r["mod"] is not None:
            mo = r["mod"](m)
        elif folder_model:
            parts = os.path.relpath(fp, root).split(os.sep)
            mo = parts[0] if parts else "?"
            mo = re.sub(r"^D\d+_", "", mo)          # strip the device prefix
        else:
            mo = "?"
        dv.setdefault(d, {"model": mo, "n": 0})["n"] += 1
    by = {}
    for d, v in dv.items(): by.setdefault(v["model"], []).append((d, v["n"]))
    print(f"\n=== {kind} @ {root} — {len(dv)} devices, {len(by)} models ===")
    ok = []
    for mo, ds in sorted(by.items()):
        ds = sorted(ds, key=lambda x: -x[1])
        if len(ds) < 2: continue
        usable = [x for x in ds if x[1] >= need]
        flag = "  <== USABLE" if len(usable) >= 2 else ""
        print(f"  {mo:32s} {len(ds)} devices, >= {need} imgs: {len(usable)}{flag}")
        print(f"      {ds}")
        if len(usable) >= 2: ok.append((mo, usable))
    return ok

# Dresden is already on your Drive — start here.
_ = screen(os.path.join(DRIVE, "forensic_datasets/dresden/Dresden_Exp"), "dresden", need=340)

## 2. Direct-download datasets (scriptable)
FODB and VISION are served over plain HTTP by university hosts. **Check the current URLs on the dataset pages before running** — paths change. Download a *subset*: you need a handful of devices, not the whole corpus.

In [ ]:
# %% 2 — FODB / VISION subset fetch (edit URLs after checking the dataset pages)
# FODB   landing: https://faui1-files.cs.fau.de/public/mmsec/datasets/fodb/
# VISION landing: https://lesc.dinfo.unifi.it/en/datasets
#
# Pull ONE device at a time and screen before pulling more; the full corpora are large.
FETCH = []   # e.g. [("https://faui1-files.cs.fau.de/public/mmsec/datasets/fodb/<file>", "fodb")]

for url, sub in FETCH:
    out = os.path.join(DEST, sub); os.makedirs(out, exist_ok=True)
    fn = os.path.join(out, os.path.basename(url))
    if os.path.exists(fn): print("have", fn); continue
    print("fetching", url)
    subprocess.run(["wget", "-c", "-q", "--show-progress", "-O", fn, url], check=False)
    if fn.endswith((".zip",".ZIP")):   subprocess.run(["unzip","-q","-n",fn,"-d",out])
    elif fn.endswith((".tar.gz",".tgz")): subprocess.run(["tar","-xzf",fn,"-C",out])
print("done")

## 3. Daxing, if you go that route
Download manually from the Baidu link (fetch code `2024`), upload the archives to `MyDrive/forensic_datasets/daxing_raw/`, then run this to unpack and screen. **Use iPhone 6S only — do not merge 6S and 6S Plus** (different sensor and lens).

In [ ]:
# %% 3 — unpack manually-uploaded Daxing archives, then screen
RAW = os.path.join(DEST, "daxing_raw"); OUT = os.path.join(DEST, "daxing")
os.makedirs(OUT, exist_ok=True)
arcs = sorted(glob.glob(os.path.join(RAW, "*.zip")) + glob.glob(os.path.join(RAW, "*.rar")))
print("archives found:", len(arcs))
if any(a.endswith(".rar") for a in arcs):
    subprocess.run(["apt-get","-qq","install","-y","unrar"], check=False)
for a in arcs:
    print("unpacking", os.path.basename(a))
    if a.endswith(".zip"): subprocess.run(["unzip","-q","-n",a,"-d",OUT])
    else:                  subprocess.run(["unrar","x","-o-","-inul",a,OUT])
if os.path.isdir(OUT) and glob.glob(os.path.join(OUT,"**","*.*"), recursive=True):
    _ = screen(OUT, "daxing", need=340, folder_model=False)
else:
    print("nothing unpacked yet — upload the Baidu archives to", RAW)

## 4. What to do with the result

Whichever dataset you land on, the design the review asks for is:

- **4–6 devices of one model** for independent adaptation (one adapter each) — this is what
  fixes the n=3-adapter bottleneck, and it matters more than more generated images.
- **Several further same-model bodies held out** solely to estimate the shared component
  `kappa_model = (1/M) * sum_d K_d`, then report device fingerprints both raw and
  orthogonalised against it.
- **Flat-field scenes for fingerprint estimation** where the dataset has them (Daxing's sky and
  white-wall; Dresden has none) and natural scenes for adaptation and held-out validation.
- **Split by scene, not by filename block**, when scene count is small.

Then point `C.DRESDEN_DIR` at the new root, swap `FNAME_RE`, give it a fresh `OUT_DIR`
(the manifest changes, so the protocol hash must too), and re-run S0 onward.